# CST research reset

This experiment implements `next_experimentations.md` on a new standard CST database. It compares functional PCA, ordinary functional diffusion maps, and quantity-aware functional diffusion maps in 2D and 3D, then checks decoded optimization candidates. Prior NACA caches are historical.


In [ ]:
from pathlib import Path
import sys, runpy
import pandas as pd
from IPython.display import Image, display
root = Path.cwd()
if not (root / "utils/cst_reduction.py").exists():
    root = root.parent
assert (root / "utils/cst_reduction.py").exists()
QOI = "min_drag"
METRIC = "balanced"
# Other quantities: mean_drag, negative_efficiency; other metric: drag.
script = root / "notebooks/cst_reset.py"
saved_argv = sys.argv
try:
    sys.argv = [str(script), "--qoi", QOI, "--metric", METRIC]
    runpy.run_path(str(script), run_name="__main__")
finally:
    sys.argv = saved_argv
run_name = f"{QOI}_{METRIC}"
data = root / "data/cst_reset" / run_name
figures = root / "figures/cst_reset" / run_name


## New dataset and confidence audit
768 directly sampled CST shapes with 15 independent variables; CD, CL, CM and pointwise confidence on 31 × 21 conditions. The raw corpus retains all cases. The primary cohort requires confidence ≥0.75 throughout the operating domain.


In [ ]:
display(pd.read_csv(root / "data/cst_reset/catalogue.csv").head())
display(pd.read_csv(data / "corpus_audit.csv").groupby("eligible").agg(n=("source_index", "size"), min_confidence=("confidence_min", "min")))
display(Image(filename=str(figures / "dataset.png")))


## Conditional projection and scalar/field tradeoff
All encoders consume known complete fields. Quantity-aware extension additionally computes the quantity from those fields. This tests preservation after reduction, not blind prediction from a new geometry. Inspect whole-field and shape loss alongside the scalar score.


In [ ]:
display(pd.read_csv(data / "projection_metrics.csv"))
display(pd.read_csv(data / "embedding_tuning.csv"))
for name in ["projection_maps", "projection_3d", "metric_tradeoff", "clusters"]:
    display(Image(filename=str(figures / f"{name}.png")))


## Feasible decoding and optimization verification
A smooth scalar surrogate is optimized over the training latent hull. Barycentric CST decoding preserves coefficient bounds and thickness; fresh solver calls check the objective, confidence and encoder round trip. The hull is not established to equal the attainable field-embedding image. Incumbents are retained if candidates degrade the objective.


In [ ]:
display(pd.read_csv(data / "optimization.csv"))
display(pd.read_csv(data / "adopted_designs.csv"))
display(pd.read_csv(data / "grid_check.csv"))
display(pd.read_csv(data / "model_size_check.csv"))
display(Image(filename=str(figures / "optimization.png")))


## Conclusion and scope
Quantity preservation is possible, but field reconstruction and inverse-map consistency are essential independent checks. The target-aware drag gain reverses with another NeuralFoil model size. See `cst_reset.md`, the dataset README and `../report/cst_reset.pdf` for the evidence and limitations.
